In [1]:
import json
import pickle
import numpy as np
import pandas as pd
import rdkit
from rdkit.Chem import AllChem
import py3Dmol
from joblib import Parallel, delayed
from ringer.utils import chem, internal_coords

In [2]:
def visualize_conformer(mol, title, id=0):
    viewer = py3Dmol.view(width=400, height=400)
    if mol.GetNumConformers() > 0:
        mol_block = rdkit.Chem.MolToMolBlock(mol, confId=id)
        viewer.addModel(mol_block, "mol")
        viewer.setStyle({"stick": {"color": "spectrum"}})
        viewer.zoomTo()
        viewer.setBackgroundColor("white")
        print(f"{title} - Conformer {id}")
        return viewer
    else:
        print(f"{title} has no conformers")
        return None

In [3]:
with open("sample/reconstructed_mols/F.Mec.q.Y.pickle", "rb") as source:
    s_o_mol = pickle.load(source)
s_o_mol.GetNumConformers()

2756

In [4]:
with open("sample/reconstructed_mols_no_opt/F.Mec.q.Y.pickle", "rb") as source:
    s_no_o_mol = pickle.load(source)
s_no_o_mol.GetNumConformers()

2756

In [5]:
def compare_ring_bond_lengths_to_training_means(
    mol,
    conf_id=0,
    means_path="assets/models/conditional/training_mean_distances.json",
):
    with open(means_path, "r") as source:
        mean_distances = json.load(source)

    macrocycle_idxs = chem.get_macrocycle_idxs(mol, n_to_c=True)
    if macrocycle_idxs is None:
        raise ValueError("No macrocycle indices found for molecule")

    bond_idxs = internal_coords.get_macrocycle_bond_idxs(macrocycle_idxs)
    conf = mol.GetConformer(int(conf_id))

    atom_labels = ["N", "Calpha", "CO"]
    rows = []
    for ring_pos, (idx1, idx2) in enumerate(bond_idxs):
        atom_label = atom_labels[ring_pos % 3]
        expected = float(mean_distances[atom_label])
        actual = float(AllChem.GetBondLength(conf, int(idx1), int(idx2)))
        delta = actual - expected
        rows.append(
            {
                "ring_bond_pos": ring_pos,
                "atom_i": int(idx1),
                "atom_j": int(idx2),
                "first_atom_label": atom_label,
                "expected_length": expected,
                "actual_length": actual,
                "delta": delta,
            }
        )

    df = pd.DataFrame(rows)
    df["abs_delta"] = df["delta"].abs()
    df["rel_abs_delta_pct"] = 100.0 * df["abs_delta"] / df["expected_length"]

    likely_broken = df.loc[df["abs_delta"].idxmax()].copy()
    return df, likely_broken, macrocycle_idxs, mean_distances

In [7]:
demo_mol = s_no_o_mol
means_path = "assets/models/conditional/training_mean_distances.json"

with open(means_path, "r") as source:
    mean_distances_demo = json.load(source)

macrocycle_idxs_raw = chem.get_macrocycle_idxs(demo_mol, n_to_c=False)
macrocycle_idxs_n2c = chem.get_macrocycle_idxs(demo_mol, n_to_c=True)
bond_idxs = internal_coords.get_macrocycle_bond_idxs(macrocycle_idxs_n2c)

print("INPUT 1 (raw ring order, n_to_c=False):")
print(macrocycle_idxs_raw)
print("\nOUTPUT 1 (canonical ring order, n_to_c=True, starts at N):")
print(macrocycle_idxs_n2c)

print("\nINPUT 2: get_macrocycle_bond_idxs(macrocycle_idxs_n2c)")
print("Input length:", len(macrocycle_idxs_n2c))
print("Input list:", macrocycle_idxs_n2c)

print("\nOUTPUT 2: wrapped adjacent bond pairs")
print(bond_idxs)
print("Last bond closes ring:", bond_idxs[-1], "==", (macrocycle_idxs_n2c[-1], macrocycle_idxs_n2c[0]))

atom_labels_cycle = ["N", "Calpha", "CO"]
rows = []
for ring_pos, (idx1, idx2) in enumerate(bond_idxs):
    label = atom_labels_cycle[ring_pos % 3]
    rows.append(
        {
            "ring_bond_pos": ring_pos,
            "atom_i(first)": int(idx1),
            "atom_j(second)": int(idx2),
            "assigned_label_from_first_atom": label,
            "mean_length_from_json": float(mean_distances_demo[label]),
            "actual_length": float(AllChem.GetBondLength(demo_mol.GetConformer(0), int(idx1), int(idx2))),
        }
    )

mapping_df = pd.DataFrame(rows)
print("\nFinal mapping used in compare_ring_bond_lengths_to_training_means:")
display(mapping_df)

INPUT 1 (raw ring order, n_to_c=False):
[1, 36, 34, 33, 27, 25, 24, 15, 13, 12, 4, 2]

OUTPUT 1 (canonical ring order, n_to_c=True, starts at N):
[33, 27, 25, 24, 15, 13, 12, 4, 2, 1, 36, 34]

INPUT 2: get_macrocycle_bond_idxs(macrocycle_idxs_n2c)
Input length: 12
Input list: [33, 27, 25, 24, 15, 13, 12, 4, 2, 1, 36, 34]

OUTPUT 2: wrapped adjacent bond pairs
[[33, 27], [27, 25], [25, 24], [24, 15], [15, 13], [13, 12], [12, 4], [4, 2], [2, 1], [1, 36], [36, 34], [34, 33]]
Last bond closes ring: [34, 33] == (34, 33)

Final mapping used in compare_ring_bond_lengths_to_training_means:


,ring_bond_pos,atom_i(first),atom_j(second),assigned_label_from_first_atom,mean_length_from_json,actual_length
0,0,33,27,N,1.455443,1.457400
1,1,27,25,Calpha,1.535209,1.536800
2,2,25,24,CO,1.338638,1.341300
3,3,24,15,N,1.455443,1.457600
4,4,15,13,Calpha,1.535209,1.536700
5,5,13,12,CO,1.338638,1.341300
6,6,12,4,N,1.455443,1.457400
7,7,4,2,Calpha,1.535209,1.536700
8,8,2,1,CO,1.338638,1.574369
9,9,1,36,N,1.455443,1.457100


In [11]:
def _compare_ring_bonds_single_conf(conf_id, bond_idxs, positions, expected_per_pos, atom_label_per_pos):
    rows = []
    for ring_pos, (idx1, idx2) in enumerate(bond_idxs):
        expected = expected_per_pos[ring_pos]
        actual = float(np.linalg.norm(positions[idx1] - positions[idx2]))
        delta = actual - expected
        rows.append(
            {
                "conf_id": int(conf_id),
                "ring_bond_pos": ring_pos,
                "atom_i": int(idx1),
                "atom_j": int(idx2),
                "first_atom_label": atom_label_per_pos[ring_pos],
                "expected_length": expected,
                "actual_length": actual,
                "delta": delta,
            }
        )
    return rows


def compare_ring_bond_lengths_all_conformers(
    mol,
    means_path="assets/models/conditional/training_mean_distances.json",
    n_jobs=-1,
):
    with open(means_path, "r") as source:
        mean_distances = json.load(source)

    macrocycle_idxs = chem.get_macrocycle_idxs(mol, n_to_c=True)
    if macrocycle_idxs is None:
        raise ValueError("No macrocycle indices found for molecule")

    bond_idxs = internal_coords.get_macrocycle_bond_idxs(macrocycle_idxs)
    if mol.GetNumConformers() == 0:
        raise ValueError("Molecule has no conformers")

    atom_labels = ["N", "Calpha", "CO"]
    atom_label_per_pos = [atom_labels[i % 3] for i in range(len(bond_idxs))]
    expected_per_pos = [float(mean_distances[lbl]) for lbl in atom_label_per_pos]

    conf_positions = [
        (conf.GetId(), np.asarray(conf.GetPositions())) for conf in mol.GetConformers()
    ]

    results = Parallel(n_jobs=n_jobs)(
        delayed(_compare_ring_bonds_single_conf)(
            conf_id, bond_idxs, positions, expected_per_pos, atom_label_per_pos
        )
        for conf_id, positions in conf_positions
    )

    df = pd.DataFrame([row for conf_rows in results for row in conf_rows])
    df["abs_delta"] = df["delta"].abs()
    df["rel_abs_delta_pct"] = 100.0 * df["abs_delta"] / df["expected_length"]

    worst_per_conf = (
        df.loc[df.groupby("conf_id")["abs_delta"].idxmax()]
        .sort_values("abs_delta", ascending=False)
        .reset_index(drop=True)
    )

    return df, worst_per_conf, macrocycle_idxs, mean_distances

In [27]:
all_bonds_df, worst_per_conf, macrocycle_idxs, _ = compare_ring_bond_lengths_all_conformers(s_no_o_mol)
worst_per_conf

,conf_id,ring_bond_pos,atom_i,atom_j,first_atom_label,expected_length,actual_length,delta,abs_delta,rel_abs_delta_pct
0,1217,5,13,12,CO,1.338638,2.038811,0.700173,0.700173,52.304912
1,1031,3,24,15,N,1.455443,0.896486,-0.558957,0.558957,38.404568
2,608,1,27,25,Calpha,1.535209,1.079992,-0.455217,0.455217,29.651797
3,1200,10,36,34,Calpha,1.535209,1.104881,-0.430329,0.430329,28.030630
4,1890,11,34,33,CO,1.338638,1.756906,0.418269,0.418269,31.245854
...,...,...,...,...,...,...,...,...,...,...
2751,1247,8,2,1,CO,1.338638,1.341400,0.002762,0.002762,0.206358
2752,1968,8,2,1,CO,1.338638,1.341400,0.002762,0.002762,0.206357
2753,1818,8,2,1,CO,1.338638,1.341400,0.002762,0.002762,0.206357
2754,2426,8,2,1,CO,1.338638,1.341400,0.002762,0.002762,0.206355


In [25]:
all_bonds_df, worst_per_conf, macrocycle_idxs, _ = compare_ring_bond_lengths_all_conformers(s_o_mol)

In [26]:
worst_per_conf

,conf_id,ring_bond_pos,atom_i,atom_j,first_atom_label,expected_length,actual_length,delta,abs_delta,rel_abs_delta_pct
0,2129,6,12,4,N,1.455443,1.459301,0.003858,0.003858,0.265068
1,1442,3,24,15,N,1.455443,1.458956,0.003513,0.003513,0.241394
2,2628,9,1,36,N,1.455443,1.458955,0.003512,0.003512,0.241292
3,398,9,1,36,N,1.455443,1.458940,0.003497,0.003497,0.240290
4,2034,9,1,36,N,1.455443,1.458927,0.003484,0.003484,0.239349
...,...,...,...,...,...,...,...,...,...,...
2751,704,8,2,1,CO,1.338638,1.341400,0.002762,0.002762,0.206350
2752,1107,8,2,1,CO,1.338638,1.341400,0.002762,0.002762,0.206349
2753,2736,8,2,1,CO,1.338638,1.341400,0.002762,0.002762,0.206349
2754,892,8,2,1,CO,1.338638,1.341400,0.002762,0.002762,0.206349


In [20]:
target_mol = s_o_mol
target_conf_id = 0

bond_df, likely_broken_row, macrocycle_idxs, training_mean_distances = compare_ring_bond_lengths_to_training_means(
    target_mol,
    conf_id=target_conf_id,
)

print(f"Macrocycle atom count: {len(macrocycle_idxs)}")
print(f"Conformer analyzed: {target_conf_id}")
display(bond_df)

print("Likely broken bond during best_dist init (max |delta|):")
display(pd.DataFrame([likely_broken_row]))


Macrocycle atom count: 12
Conformer analyzed: 0


,ring_bond_pos,atom_i,atom_j,first_atom_label,expected_length,actual_length,delta,abs_delta,rel_abs_delta_pct
0,0,33,27,N,1.455443,1.457400,0.001957,0.001957,0.134465
1,1,27,25,Calpha,1.535209,1.536800,0.001591,0.001591,0.103608
2,2,25,24,CO,1.338638,1.341300,0.002662,0.002662,0.198892
3,3,24,15,N,1.455443,1.457600,0.002157,0.002157,0.148195
4,4,15,13,Calpha,1.535209,1.536700,0.001491,0.001491,0.097089
5,5,13,12,CO,1.338638,1.341300,0.002662,0.002662,0.198889
6,6,12,4,N,1.455443,1.457400,0.001957,0.001957,0.134463
7,7,4,2,Calpha,1.535209,1.536700,0.001491,0.001491,0.097094
8,8,2,1,CO,1.338638,1.341400,0.002762,0.002762,0.206358
9,9,1,36,N,1.455443,1.457889,0.002446,0.002446,0.168089


Likely broken bond during best_dist init (max |delta|):


,ring_bond_pos,atom_i,atom_j,first_atom_label,expected_length,actual_length,delta,abs_delta,rel_abs_delta_pct
8,8,2,1,CO,1.338638,1.3414,0.002762,0.002762,0.206358


In [21]:
def visualize_conformer_with_broken_bond(
    mol,
    title,
    conf_id=0,
    broken_bond=None,
    label_text=None,
    width=700,
    height=550,
):
    viewer = py3Dmol.view(width=width, height=height)
    if mol.GetNumConformers() == 0:
        print(f"{title} has no conformers")
        return None

    mol_block = rdkit.Chem.MolToMolBlock(mol, confId=int(conf_id))
    viewer.addModel(mol_block, "mol")
    viewer.setStyle({}, {"stick": {"color": "lightgray", "radius": 0.12}})

    if broken_bond is not None:
        idx1, idx2 = map(int, broken_bond)
        viewer.setStyle(
            {"index": [idx1, idx2]},
            {"stick": {"color": "red", "radius": 0.26}, "sphere": {"color": "red", "radius": 0.36}},
        )

        conf = mol.GetConformer(int(conf_id))
        p1 = conf.GetAtomPosition(idx1)
        p2 = conf.GetAtomPosition(idx2)
        start = {"x": float(p1.x), "y": float(p1.y), "z": float(p1.z)}
        end = {"x": float(p2.x), "y": float(p2.y), "z": float(p2.z)}
        mid = {
            "x": 0.5 * (start["x"] + end["x"]),
            "y": 0.5 * (start["y"] + end["y"]),
            "z": 0.5 * (start["z"] + end["z"]),
        }

        viewer.addLine({"start": start, "end": end, "color": "red", "linewidth": 6})
        viewer.addLabel(
            label_text if label_text is not None else f"Likely broken init bond: {idx1}-{idx2}",
            {
                "position": mid,
                "fontSize": 13,
                "backgroundColor": "black",
                "fontColor": "white",
                "inFront": True,
            },
        )
        viewer.zoomTo({"index": [idx1, idx2]})

    viewer.setBackgroundColor("white")
    print(f"{title} - Conformer {conf_id}")
    return viewer


broken_bond = (int(likely_broken_row["atom_i"]), int(likely_broken_row["atom_j"]))
label = (
    f"Likely broken init bond {broken_bond[0]}-{broken_bond[1]} | "
    f"actual={likely_broken_row['actual_length']:.3f} A, "
    f"expected={likely_broken_row['expected_length']:.3f} A, "
    f"delta={likely_broken_row['delta']:+.3f} A"
)

viewer_broken = visualize_conformer_with_broken_bond(
    target_mol,
    title="s_no_o_mol (likely broken bond highlighted)",
    conf_id=target_conf_id,
    broken_bond=broken_bond,
    label_text=label,
)
if viewer_broken:
    viewer_broken.show()


s_no_o_mol (likely broken bond highlighted) - Conformer 0


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [22]:
viewer2 = visualize_conformer(s_no_o_mol, "s_no_o_mol", id=1475)
if viewer2:
    viewer2.show()

s_no_o_mol - Conformer 1475


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [23]:
viewer1 = visualize_conformer(s_o_mol, "s_o_mol", id=1475)
if viewer1:
    viewer1.show()

s_o_mol - Conformer 1475


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

Bad pipe message: %s [b'qj\xdb\xc0\xb7QJQ\xca\x88\xa8@N.\x01C\x06_\x00\x02\xbc\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x10\x00\x11\x00\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18\x00\x19\x00\x1a\x00\x1b\x00\x1e\x00\x1f\x00 \x00!\x00"\x00#\x00$\x00%\x00&\x00\'\x00(\x00)\x00*\x00+\x00,\x00-\x00.\x00']
Bad pipe message: %s [b'0\x001\x002\x003\x004\x005\x006\x007\x008\x009\x00:\x00;\x00<\x00=\x00>\x00?\x00@\x00A\x00B\x00C\x00D\x00E\x00F\x00']
Bad pipe message: %s [b'h\x00i\x00j\x00k\x00l\x00m\x00\x84\x00\x85\x00\x86\x00\x87\x00\x88\x00\x89\x00\x8a\x00\x8b\x00\x8c\x00\x8d\x00\x8e\x00\x8f\x00\x90\x00\x91\x00\x92\x00\x93\x00\x94\x00\x95\x00\x96\x00\x97\x00\x98\x00\x99\x00\x9a\x00\x9b\x00\x9c\x00\x9d\x00\x9e\x00\x9f\x00\xa0\x00\xa1\x00\xa2\x00\xa3\x00\xa4\x00\xa5\x00\xa6\x00\xa7\x00\xa8\x00\xa9\x00\xaa\x00\xab\x00\xac\x00\xad\x00\xae\x00\xaf\x00\xb0\x00']
Bad pipe message: %s [b'9\x02\xcdTh\xab\x7f\xd